Ringkasan Alur Kerja
Dokumen ini akan memandu Anda melalui proses lengkap untuk menganalisis data menggunakan BigQuery dan Model Bahasa (LLM) dari Google. Proses ini dibagi menjadi empat tujuan utama:

1. **Ingest Data ke BigQuery**: Kita akan memulai dengan memuat data dari file CSV, membersihkannya, lalu memasukkannya ke dalam tabel BigQuery. Ini adalah fondasi agar data kita dapat diakses oleh layanan Google Cloud lainnya.

2. **Query Bahasa Alami**: Setelah data ada di BigQuery, kita akan menggunakan LangChain dan model AI Gemini untuk mengajukan pertanyaan ke data kita menggunakan bahasa manusia biasa, tanpa perlu menulis query SQL secara manual.

3. **Pembuatan Ringkasan Otomatis**: Kita akan mendemonstrasikan cara mengajukan serangkaian pertanyaan secara otomatis untuk mengumpulkan wawasan kunci dari data. Kemudian, model AI akan merangkum semua jawaban tersebut menjadi sebuah laporan naratif.

4. **Chatbot Interaktif**: Sebagai puncaknya, kita akan membangun sebuah chatbot sederhana yang memungkinkan Anda untuk berinteraksi dan "bercakap-cakap" dengan data Anda secara real-time.

Mari kita mulai dengan langkah pertama: persiapan lingkungan kerja.

# Langkah 1: Pengaturan Awal, Instalasi, dan Autentikasi

*Cell* dibawah ini bertujuan untuk mempersiapkan lingkungan kerja Anda di Google Colab dengan menginstal semua *library* yang diperlukan.

### 1.1 Instalasi Library yang Dibutuhkan
Perintah ini menginstal semua library Python yang diperlukan dari LangChain dan Google Cloud.

Cukup jalankan sel ini sekali saja setiap kali Anda membuka notebook.

In [1]:
# Instalasi Library yang Dibutuhkan

!pip install --quiet google-cloud-aiplatform google-cloud-bigquery pandas db-dtypes langchain-google-vertexai langchain-community langchain-experimental sqlalchemy-bigquery
print("✅ Instalasi selesai.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.9/438.9 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 736.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.5 MB/s eta 0:00:00
✅ Instalasi selesai.


`!pip install --quiet ...`: Perintah ini menggunakan `pip` (manajer paket Python) untuk menginstal beberapa *library* sekaligus. Tanda ! menunjukkan bahwa perintah ini dijalankan di *shell* (terminal) bukan di kernel Python. Opsi --quiet menekan output instalasi yang panjang, hanya menampilkan pesan penting.

**Library yang Digunakan:**

- `google-cloud-aiplatform:` SDK resmi dari Google untuk berinteraksi dengan Vertex AI. Digunakan untuk menginisialisasi layanan dan memanggil model AI seperti Gemini.

- `google-cloud-bigquery`: Library klien resmi dari Google untuk berinteraksi dengan BigQuery, layanan data warehouse tanpa server dari Google.

- `pandas`: Library fundamental untuk analisis dan manipulasi data di Python. Digunakan untuk membaca file CSV dan mengubahnya menjadi struktur data yang disebut DataFrame.

- `db-dtypes`: Menyediakan tipe data khusus untuk database (seperti BigQuery) agar dapat digunakan dengan lancar oleh Pandas.

- `langchain-google-vertexai`: Bagian dari ekosistem LangChain yang berfungsi sebagai "jembatan" untuk menghubungkan LangChain dengan model-model AI yang ada di Google Vertex AI.

- `langchain-community & langchain-experimental`: Berisi berbagai komponen, tool, dan agen yang dikontribusikan oleh komunitas LangChain. Kita menggunakannya untuk create_sql_agent.

- `sqlalchemy-bigquery`: Library yang memungkinkan SQLAlchemy (sebuah toolkit SQL populer di Python) untuk "berbicara" dengan BigQuery. LangChain menggunakan SQLAlchemy di belakang layar untuk berinteraksi dengan berbagai jenis database SQL.

## 1.2 Autentikasi Akun Google
Menjalankan sel ini akan memunculkan pop-up untuk login ke akun Google Anda.

Ini memberikan izin kepada notebook untuk mengakses layanan Google Cloud Anda.

In [2]:
# Autentikasi Akun Google Anda

from google.colab import auth
import sys

if "google.colab" in sys.modules:
    auth.authenticate_user()
    print("✅ Autentikasi berhasil.")

✅ Autentikasi berhasil.


`from google.colab import auth:` Mengimpor modul auth dari library google.colab yang khusus tersedia di lingkungan Google Colaboratory.

## 1.3 Inisialisasi Proyek dan Klien
Atur variabel untuk Project ID dan Lokasi Anda, kemudian inisialisasi Vertex AI SDK dan Klien BigQuery.

In [3]:
import vertexai
from google.cloud import bigquery

# Atur ID Proyek Anda
PROJECT_ID = "eikon-dev-ai-team"  # @param {type:"string"}

# Atur Lokasi Anda
# Vertex AI Search tersedia di multi-region "global", "eu", dan "us".
LOCATION = "global"  # @param {type:"string"}

# Inisialisasi Vertex AI SDK
vertexai.init(project=PROJECT_ID, location=LOCATION)

# Inisialisasi klien BigQuery
bq_client = bigquery.Client(project=PROJECT_ID)

print(f"Vertex AI SDK telah diinisialisasi untuk proyek: {PROJECT_ID}")
print(f"Klien BigQuery telah diinisialisasi untuk proyek: {PROJECT_ID}")

Vertex AI SDK telah diinisialisasi untuk proyek: eikon-dev-ai-team
Klien BigQuery telah diinisialisasi untuk proyek: eikon-dev-ai-team


## 1.4 (Opsional) Mounting Google Drive untuk Service Account
Jika Anda menggunakan service account, Anda bisa mount Google Drive
dan mengatur path ke file JSON kredensial Anda.

In [4]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = userdata.get('SERVICE_ACCOUNT_PATH')

Mounted at /content/drive


# Langkah 2: Memuat dan Mempersiapkan Data

Tujuan cell ini adalah membaca data mentah dari file CSV, membersihkannya dari baris yang tidak relevan, dan memperbaiki format nama kolom agar sesuai dengan standar BigQuery.

### 2.1 Muat File CSV ke Pandas DataFrame
Pastikan file `.csv` sudah diunggah ke lingkungan Colab. Delimiter diatur ke titik koma (';').

In [5]:
import pandas as pd
import io

nama_file = "data_pelatihan.csv" #@param {type:"string"}
try:
    # Muat file CSV.
    df = pd.read_csv(f'{nama_file}', delimiter=';')

    print(f"Berhasil memuat {nama_file}.")

    # Hapus semua baris yang seluruh kolomnya kosong
    df.dropna(how='all', inplace=True)

    print("\nBerhasil membersihkan baris yang sepenuhnya kosong.")
    print("Pratinjau Data:")
    display(df.head())

except FileNotFoundError:
    print(f"ERROR: File '{nama_file}' tidak ditemukan. Mohon unggah file tersebut ke lingkungan Colab menggunakan panel file di sebelah kiri.")
except Exception as e:
    print(f"Terjadi kesalahan: {e}")

Berhasil memuat data_pelatihan.csv.

Berhasil membersihkan baris yang sepenuhnya kosong.
Pratinjau Data:


,nama_peserta,no_ktp,jenis_kelamin_peserta,usia_peserta,pendidikan_peserta,jurusan_peserta,institusi_peserta,email,alamat_peserta,kabkot_peserta,...,nama_kegiatan,nama_lokasi,wilayah_kerja,stt_lokasi,tahun_pelatihan,tanggal_mulai_pel,no_sertifikat_peserta,kompeten,id_pelatihan,tahun_lahir_peserta
0,"DRS. SUDIYO UTOMO, M.SI",5371**********01,Laki-laki,67 tahun,STRATA II,PSDAL,UNDANA,di****@gmail.com,******,Kota Kupang,...,"TOT (Instruktur, Assesor, Peralatan)",HOTEL SWISS BELL IN,NUSA TENGGARA TIMUR,Luar Balai,2018.0,16/04/2018 00:00:00,00076551/NTT/PLU/REG/2018/,KOMPETEN,3838.0,1957.0
1,"IR. DOLLY W. KARELS, M.SI",5371**********03,Perempuan,62 tahun,STRATA II,ILMU LINGKUNGAN,UNDANA,do****@gmail.com,******,Kota Kupang,...,"TOT (Instruktur, Assesor, Peralatan)",HOTEL SWISS BELL IN,NUSA TENGGARA TIMUR,Luar Balai,2018.0,16/04/2018 00:00:00,00076554/NTT/PLU/REG/2018/,KOMPETEN,3838.0,1962.0
2,"JULIANTI MARBUN, ST, M.SC",6171**********01,Perempuan,39 tahun,STRATA II,ILMU LINGKUNGAN,UGM,ju****@gmail.com,******,Kota Pontianak,...,"TOT (Instruktur, Assesor, Peralatan)",HOTEL SWISS BELL IN,NUSA TENGGARA TIMUR,Luar Balai,2018.0,16/04/2018 00:00:00,00076555/NTT/PLU/REG/2018/,KOMPETEN,3838.0,1985.0
3,"ROSMIYATI A. BELLA, ST., MT",5371**********01,Perempuan,48 tahun,STRATA II,TEKNIK SIPIL,ITB,ga****@yahoo.com,******,Kota Kupang,...,"TOT (Instruktur, Assesor, Peralatan)",HOTEL SWISS BELL IN,NUSA TENGGARA TIMUR,Luar Balai,2018.0,16/04/2018 00:00:00,00076563/NTT/PLU/REG/2018/,KOMPETEN,3838.0,1976.0
4,"ZOFAR A. BANUNAEK, ST.,MT",5371**********02,Laki-laki,34 tahun,STRATA II,TEK. LINGKUNGAN,ITS,zo****@gmail.com,******,Kota Kupang,...,"TOT (Instruktur, Assesor, Peralatan)",HOTEL SWISS BELL IN,NUSA TENGGARA TIMUR,Luar Balai,2018.0,16/04/2018 00:00:00,00076568/NTT/PLU/REG/2018/,KOMPETEN,3838.0,1990.0


In [6]:
len(df)

2999

**Penjelasan Kode:**

1. `import pandas as pd`: Mengimpor library pandas dan memberinya alias pd agar lebih mudah dipanggil.

2. `df = pd.read_csv(...)`: Membaca file data_pelatihan.csv. Argumen delimiter=';' memberi tahu pandas bahwa kolom-kolom dalam file ini dipisahkan oleh titik koma, bukan koma (standar). Hasilnya disimpan dalam sebuah DataFrame bernama df.

3. `df.dropna(how='all', inplace=True)`: Menghapus baris dari DataFrame. Opsi how='all' berarti baris hanya akan dihapus jika semua nilainya kosong. inplace=True berarti perubahan langsung diterapkan pada DataFrame df tanpa perlu membuatnya kembali (df = ...).

### 2.2. Bersihkan Nama Kolom untuk BigQuery
BigQuery memiliki aturan penamaan kolom yang lebih ketat daripada pandas (misalnya, tidak boleh ada spasi, karakter spesial). Sel ini membersihkan nama kolom agar kompatibel.

In [7]:
def clean_bq_column_names(df):
    """Membersihkan nama kolom DataFrame agar kompatibel dengan BigQuery."""
    new_columns = []
    for col in df.columns:
        # Ganti spasi dan karakter spesial dengan garis bawah (underscore)
        new_col = ''.join(e if e.isalnum() else '_' for e in col)
        # Hapus garis bawah di awal/akhir dan yang berurutan
        new_col = new_col.strip('_')
        new_col = '_'.join(filter(None, new_col.split('_')))
        # Pastikan tidak diawali dengan angka
        if new_col and new_col[0].isdigit():
            new_col = '_' + new_col
        new_columns.append(new_col)
    df.columns = new_columns
    return df

if 'df' in locals():

    df = clean_bq_column_names(df)


    print("Pratinjau data:")
    display(df.head())

Pratinjau data:


,nama_peserta,no_ktp,jenis_kelamin_peserta,usia_peserta,pendidikan_peserta,jurusan_peserta,institusi_peserta,email,alamat_peserta,kabkot_peserta,...,nama_kegiatan,nama_lokasi,wilayah_kerja,stt_lokasi,tahun_pelatihan,tanggal_mulai_pel,no_sertifikat_peserta,kompeten,id_pelatihan,tahun_lahir_peserta
0,"DRS. SUDIYO UTOMO, M.SI",5371**********01,Laki-laki,67 tahun,STRATA II,PSDAL,UNDANA,di****@gmail.com,******,Kota Kupang,...,"TOT (Instruktur, Assesor, Peralatan)",HOTEL SWISS BELL IN,NUSA TENGGARA TIMUR,Luar Balai,2018.0,16/04/2018 00:00:00,00076551/NTT/PLU/REG/2018/,KOMPETEN,3838.0,1957.0
1,"IR. DOLLY W. KARELS, M.SI",5371**********03,Perempuan,62 tahun,STRATA II,ILMU LINGKUNGAN,UNDANA,do****@gmail.com,******,Kota Kupang,...,"TOT (Instruktur, Assesor, Peralatan)",HOTEL SWISS BELL IN,NUSA TENGGARA TIMUR,Luar Balai,2018.0,16/04/2018 00:00:00,00076554/NTT/PLU/REG/2018/,KOMPETEN,3838.0,1962.0
2,"JULIANTI MARBUN, ST, M.SC",6171**********01,Perempuan,39 tahun,STRATA II,ILMU LINGKUNGAN,UGM,ju****@gmail.com,******,Kota Pontianak,...,"TOT (Instruktur, Assesor, Peralatan)",HOTEL SWISS BELL IN,NUSA TENGGARA TIMUR,Luar Balai,2018.0,16/04/2018 00:00:00,00076555/NTT/PLU/REG/2018/,KOMPETEN,3838.0,1985.0
3,"ROSMIYATI A. BELLA, ST., MT",5371**********01,Perempuan,48 tahun,STRATA II,TEKNIK SIPIL,ITB,ga****@yahoo.com,******,Kota Kupang,...,"TOT (Instruktur, Assesor, Peralatan)",HOTEL SWISS BELL IN,NUSA TENGGARA TIMUR,Luar Balai,2018.0,16/04/2018 00:00:00,00076563/NTT/PLU/REG/2018/,KOMPETEN,3838.0,1976.0
4,"ZOFAR A. BANUNAEK, ST.,MT",5371**********02,Laki-laki,34 tahun,STRATA II,TEK. LINGKUNGAN,ITS,zo****@gmail.com,******,Kota Kupang,...,"TOT (Instruktur, Assesor, Peralatan)",HOTEL SWISS BELL IN,NUSA TENGGARA TIMUR,Luar Balai,2018.0,16/04/2018 00:00:00,00076568/NTT/PLU/REG/2018/,KOMPETEN,3838.0,1990.0


**Penjelasan Kode:**

1. `def clean_bq_column_names(...)`: Mendefinisikan sebuah fungsi untuk membersihkan nama kolom. Fungsi ini melakukan iterasi pada setiap nama kolom, mengganti spasi dan karakter non-alfanumerik dengan garis bawah (_), dan memastikan tidak ada nama yang diawali dengan angka. Ini penting karena BigQuery memiliki aturan penamaan yang ketat.

2. `df = clean_bq_column_names(df): `Memanggil fungsi yang baru dibuat untuk membersihkan nama-nama kolom di DataFrame df.


# 3. Ingest Data to BigQuery (Masukkan Data ke BigQuery)

Setelah data bersih dan siap, cell ini bertugas untuk membuat "wadah" di BigQuery (dataset dan tabel) dan mengunggah data dari DataFrame ke dalamnya.


### 3.1 Tentukan Nama Dataset dan Tabel
Dataset adalah kontainer untuk tabel Anda di BigQuery.
👇 Atur nama Dataset dan Tabel BigQuery Anda di bawah ini:

In [8]:
BQ_DATASET_ID = "dataset_pelatihan"  # @param {type:"string"}
BQ_TABLE_ID = "tabel_data_pelatihan"  # @param {type:"string"}
TABLE_REF = f"{PROJECT_ID}.{BQ_DATASET_ID}.{BQ_TABLE_ID}"

Penjelasan Kode:

1. `BQ_DATASET_ID` = ...: Mendefinisikan nama untuk dataset Anda. Dataset di BigQuery mirip seperti folder atau skema yang berisi sekumpulan tabel.

### 3.2. Buat Dataset BigQuery (jika belum ada)

In [9]:
try:
    # ID lengkap diperlukan untuk membuat dataset.
    dataset_id_full = f"{PROJECT_ID}.{BQ_DATASET_ID}"
    dataset = bigquery.Dataset(dataset_id_full)
    # Tentukan lokasi untuk dataset.
    dataset.location = "asia-southeast2" # Lokasi untuk Jakarta
    dataset = bq_client.create_dataset(dataset, exists_ok=True)
    print(f"Berhasil membuat atau menemukan dataset BigQuery: {dataset.dataset_id}")
except Exception as e:
    print(f"Terjadi kesalahan saat membuat dataset: {e}")

Berhasil membuat atau menemukan dataset BigQuery: dataset_pelatihan


**Penjelasan Kode:**


1. `dataset = bigquery.Dataset(...)`: Membuat objek dataset secara lokal.

2. `dataset.location = "asia-southeast2"`: Menentukan lokasi geografis tempat dataset akan dibuat (dalam hal ini, Jakarta). Memilih lokasi yang dekat dapat mengurangi latensi.

3. `bq_client.create_dataset(..., exists_ok=True)`: Mengirim permintaan ke Google Cloud untuk membuat dataset. Opsi exists_ok=True mencegah error jika dataset dengan nama yang sama sudah ada.

### 3.3. Unggah DataFrame ke Tabel BigQuery
Proses ini akan memuat data dari DataFrame pandas ke tabel BigQuery.
'WRITE_TRUNCATE' berarti tabel akan ditimpa jika sudah ada.

In [10]:
if 'df' in locals():
    try:
        # Konfigurasi job untuk menimpa tabel jika sudah ada
        job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")

        job = bq_client.load_table_from_dataframe(
            df, TABLE_REF, job_config=job_config
        )
        job.result()  # Tunggu hingga job selesai

        print(f"Berhasil memuat data ke tabel BigQuery: {TABLE_REF}")
        table = bq_client.get_table(TABLE_REF)
        print(f"Tabel berisi {table.num_rows} baris.")
    except Exception as e:
        print(f"Terjadi kesalahan saat mengunggah data ke BigQuery: {e}")

Berhasil memuat data ke tabel BigQuery: eikon-dev-ai-team.dataset_pelatihan.tabel_data_pelatihan
Tabel berisi 2999 baris.


**Penjelasan Kode:**


1. `bq_client.load_table_from_dataframe(...)`: Perintah utama untuk mengunggah data dari DataFrame (df) ke tabel BigQuery yang ditentukan oleh TABLE_REF.

2. `job.result()`: Menunggu hingga proses unggah (yang berjalan di background) selesai sebelum melanjutkan ke baris kode berikutnya.

---

# Langkah 4: Mengajukan Pertanyaan ke Data Menggunakan Bahasa Alami (Natural Language)

Ini adalah inti dari aplikasi AI kita. Cell ini menyiapkan "agen cerdas" yang dapat memahami pertanyaan dalam bahasa manusia, mengubahnya menjadi kode SQL, menjalankannya di BigQuery, dan mengembalikan jawabannya.

### 4.1 Inisialisasi Database, LLM, dan Agent
Langkah ini adalah inti dari sistem kita, menghubungkan LLM dengan database.

In [12]:
# 4.1 Inisialisasi Database, LLM, dan Agent
from langchain_google_vertexai import ChatVertexAI
from langchain_community.agent_toolkits import create_sql_agent
from langchain_community.utilities import SQLDatabase

model_name = "gemini-2.5-flash"  # @param {type:"string"}
try:
    # Membuat koneksi ke database BigQuery via LangChain
    db = SQLDatabase.from_uri(f"bigquery://{PROJECT_ID}/{BQ_DATASET_ID}")
    print("🔌 Koneksi ke BigQuery berhasil dibuat.")

    # Menginisialisasi model bahasa Gemini dari Vertex AI
    llm = ChatVertexAI(model_name=model_name, temperature=0)
    print(f"🧠 Model Bahasa (LLM) {model_name} berhasil diinisialisasi.")

    # Membuat SQL Agent yang akan menerjemahkan pertanyaan ke query SQL
    agent_executor = create_sql_agent(llm, db=db, agent_type="openai-tools", verbose=True)
    print("🤖 SQL Agent siap digunakan.")
except NameError as e:
    print(f"❌ Variabel penting belum terdefinisi (misal: PROJECT_ID). Error: {e}")
except Exception as e:
    print(f"❌ Terjadi kesalahan saat inisialisasi: {e}")

🔌 Koneksi ke BigQuery berhasil dibuat.
🧠 Model Bahasa (LLM) gemini-2.5-flash berhasil diinisialisasi.
🤖 SQL Agent siap digunakan.


**Penjelasan Kode:**

1. `db = SQLDatabase.from_uri(...)`: Membuat koneksi ke database BigQuery menggunakan LangChain. LangChain akan menggunakan koneksi ini untuk "melihat" skema tabel (nama kolom dan tipe datanya) guna membantu LLM membuat query yang akurat.

2. `llm = ChatVertexAI(...)`: Menginisialisasi model bahasa (LLM) Gemini. temperature=0 membuat jawaban model lebih deterministik dan faktual, cocok untuk analisis data.

3. `agent_executor = create_sql_agent(...)`: Ini adalah fungsi kunci dari LangChain. Fungsi ini "merakit" sebuah agen dengan menggabungkan tiga komponen:

  - LLM (`llm`): Otak yang mampu memahami bahasa dan menulis SQL.

  - Database Toolkit (`db`): Alat yang memberi LLM kemampuan untuk melihat skema tabel dan menjalankan query.

  - Agent Type: Menentukan cara kerja agen.
Hasilnya adalah `agent_executor` yang siap menerima perintah.

### 4.2 Ajukan Pertanyaan Anda
Tulis pertanyaan Anda dalam bahasa alami, dan agent akan mencari jawabannya.

In [13]:
# 4.2 Ajukan Pertanyaan Anda
# @markdown #### 👇 Tulis pertanyaan Anda di sini:
natural_language_query = "Berapa Total Peserta?" #@param {type:"string"}

if 'agent_executor' in locals() and natural_language_query:
    print(f"\n🤔 Pertanyaan Anda: {natural_language_query}")
    print("\n🚀 Agent sedang bekerja...")
    try:
        # Menjalankan agent dengan query dari pengguna
        result = agent_executor.invoke({"input": natural_language_query})

        print("\n\n✅ Hasil Akhir:")
        # .get("output") digunakan untuk mengambil hasil dengan aman
        print(result.get("output", "Tidak ada hasil yang ditemukan."))
    except Exception as e:
        print(f"\n❌ Terjadi error: {e}")
else:
    print("⚠️ Silakan masukkan pertanyaan yang valid dan pastikan agent sudah diinisialisasi.")


🤔 Pertanyaan Anda: Berapa Total Peserta?

🚀 Agent sedang bekerja...


> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{'tool_input': ''}`
responded:  Finally, I will construct the query to answer the question.


tabel_data_pelatihan
Invoking: `sql_db_schema` with `{'table_names': 'tabel_data_pelatihan'}`



CREATE TABLE `tabel_data_pelatihan` (
	`nama_peserta` STRING, 
	`no_ktp` STRING, 
	`jenis_kelamin_peserta` STRING, 
	`usia_peserta` STRING, 
	`pendidikan_peserta` STRING, 
	`jurusan_peserta` STRING, 
	`institusi_peserta` STRING, 
	`email` STRING, 
	`alamat_peserta` STRING, 
	`kabkot_peserta` STRING, 
	`no_hp` STRING, 
	`utusan_peserta` STRING, 
	`nama_jabatan_kerja_peserta` STRING, 
	`jenis_pelatihan` STRING, 
	`nama_balai` STRING, 
	`nama_kegiatan` STRING, 
	`nama_lokasi` STRING, 
	`wilayah_kerja` STRING, 
	`stt_lokasi` STRING, 
	`tahun_pelatihan` FLOAT64, 
	`tanggal_mulai_pel` STRING, 
	`no_sertifikat_peserta` STRING, 
	`kompeten` STRING, 
	`id_pe

**Penjelasan Kode:**

1. `result = agent_executor.invoke(...`): Mengirimkan pertanyaan (`natural_language_query`) ke agen untuk diproses. Agen akan melakukan "rantai pemikiran" (*chain of thought*): menganalisis pertanyaan, menulis *query* SQL, menjalankannya, melihat hasilnya, dan merumuskan jawaban akhir dalam bahasa manusia.

Pertanyaan yang sudah dicoba:
-  Ada berapa total baris data dalam tabel ini?

- Berapa rata-rata jumlah usia peserta?

- Berapa jumlah peserta yang berkuliah di ITS?

- Peserta paling banyak berasal dari daerah mana ?



---



# Langkah 5: Membuat Ringkasan Otomatis dari Tanya Jawab

Cell ini menunjukkan kasus penggunaan yang lebih canggih, yaitu menjalankan serangkaian analisis secara otomatis dan kemudian meminta AI untuk merangkum hasilnya menjadi laporan.

In [14]:
from tqdm.auto import tqdm
from IPython.display import display, Markdown
from langchain_google_vertexai import ChatVertexAI

### 5.1 Definisikan Daftar Pertanyaan untuk Analisis

In [18]:
list_of_questions = [
    "Berapa total jumlah peserta dalam dataset?",
    "Berapa rata-rata usia peserta?",
    "Sebutkan 3 institusi peserta yang paling umum.",
    "Berapa jumlah peserta laki-laki dan perempuan?",
    "Apa saja 5 jurusan peserta yang paling banyak muncul?"
]

**Penjelasan Kode:**

1. `list_of_questions = [...]`: Mendefinisikan daftar pertanyaan analitis yang ingin diajukan.

### 5.2 Jalankan Agent untuk Setiap Pertanyaan

In [15]:
# Jalankan Agent untuk Setiap Pertanyaan
print(f"\n🚀 Akan menjalankan agent untuk {len(list_of_questions)} pertanyaan...")

qa_results = []

if 'agent_executor' in locals():
    # Menggunakan tqdm untuk menampilkan progress bar yang informatif
    for question in tqdm(list_of_questions, desc="Mengajukan Pertanyaan"):
        print(f"\n🤔 Menanyakan: {question}")
        try:
            result = agent_executor.invoke({"input": question})
            answer = result.get("output", "Tidak ada jawaban ditemukan.")
            qa_results.append({"pertanyaan": question, "jawaban": answer})
            print(f"💡 Jawaban Agent: {answer}")
        except Exception as e:
            print(f"❌ Error saat menanyakan '{question}': {e}")
            qa_results.append({"pertanyaan": question, "jawaban": f"Error: {e}"})
    print("\n\n✅ Semua pertanyaan telah selesai diajukan.")
else:
    print("⚠️ SQL Agent belum siap. Harap jalankan langkah 4 terlebih dahulu.")


🚀 Akan menjalankan agent untuk 5 pertanyaan...


Mengajukan Pertanyaan:   0%|          | 0/5 [00:00<?, ?it/s]


🤔 Menanyakan: Berapa total jumlah peserta dalam dataset?


> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{'tool_input': ''}`
responded:  Finally, I will construct the query to answer the question.


tabel_data_pelatihan
Invoking: `sql_db_schema` with `{'table_names': 'tabel_data_pelatihan'}`



CREATE TABLE `tabel_data_pelatihan` (
	`nama_peserta` STRING, 
	`no_ktp` STRING, 
	`jenis_kelamin_peserta` STRING, 
	`usia_peserta` STRING, 
	`pendidikan_peserta` STRING, 
	`jurusan_peserta` STRING, 
	`institusi_peserta` STRING, 
	`email` STRING, 
	`alamat_peserta` STRING, 
	`kabkot_peserta` STRING, 
	`no_hp` STRING, 
	`utusan_peserta` STRING, 
	`nama_jabatan_kerja_peserta` STRING, 
	`jenis_pelatihan` STRING, 
	`nama_balai` STRING, 
	`nama_kegiatan` STRING, 
	`nama_lokasi` STRING, 
	`wilayah_kerja` STRING, 
	`stt_lokasi` STRING, 
	`tahun_pelatihan` FLOAT64, 
	`tanggal_mulai_pel` STRING, 
	`no_sertifikat_peserta` STRING, 
	`kompeten` STRING, 
	`id_pelatihan` FL

**Penjelasan Kode:**

1. `for question in tqdm(...)`: Melakukan loop untuk setiap pertanyaan dalam daftar. tqdm adalah library yang secara otomatis membuat progress bar (bilah kemajuan) yang visualnya menarik, sehingga Anda bisa memantau prosesnya.

2. `qa_results.append(...)`: Setiap pasangan pertanyaan dan jawaban dari agen disimpan dalam sebuah list bernama qa_results.

3. `context_for_summary = "\n".join(...)`: Menggabungkan semua hasil tanya jawab menjadi satu blok teks besar. Teks ini akan dijadikan "konteks" atau "informasi latar belakang" untuk LLM pada langkah berikutnya.

4. `synthesis_prompt = f"""..."""`: Membuat prompt (perintah) yang sangat spesifik untuk LLM. Prompt ini memberikan peran kepada LLM ("Anda adalah seorang analis data senior"), memberikan instruksi yang jelas ("susunlah sebuah laporan naratif"), dan menyertakan data konteks yang sudah disiapkan.

### 5.3 Buat Laporan Ringkasan dari Hasil Tanya Jawab

In [17]:
# Buat Laporan Ringkasan dari Hasil Tanya Jawab
if qa_results:
    print("\n\n✍️  Membuat laporan ringkasan dari hasil tanya-jawab...")

    # Menggabungkan semua hasil QA menjadi satu konteks untuk LLM
    context_for_summary = "\n".join([f"- Pertanyaan: {item['pertanyaan']}\n- Jawaban: {item['jawaban']}" for item in qa_results])

    # Membuat prompt yang jelas untuk LLM agar menghasilkan ringkasan naratif
    synthesis_prompt = f"""
    Anda adalah seorang analis data senior yang bertugas menulis ringkasan eksekutif.
    Berdasarkan kumpulan informasi tanya jawab berikut, susunlah sebuah laporan naratif yang kohesif dalam satu paragraf.
    Jangan hanya mengulang pertanyaan dan jawaban, tetapi rangkailah informasi tersebut menjadi sebuah cerita yang mengalir tentang profil peserta.

    Informasi yang Tersedia:
    ---
    {context_for_summary}
    ---

    Laporan Eksekutif (dalam Bahasa Indonesia):
    """

    try:
        # Menggunakan model yang lebih kuat untuk sintesis
        synthesis_llm = ChatVertexAI(model_name="gemini-2.5-pro", temperature=0.2, project=PROJECT_ID)
        final_report = synthesis_llm.invoke(synthesis_prompt)

        print("\n\n🎉 Laporan Akhir Berhasil Dibuat!")
        print("===================================")
        display(Markdown(final_report.content))
        print("===================================")

    except Exception as e:
        print(f"\n❌ Terjadi error saat membuat laporan akhir: {e}")
else:
    print("⚠️ Tidak ada hasil tanya jawab untuk diringkas.")



✍️  Membuat laporan ringkasan dari hasil tanya-jawab...


🎉 Laporan Akhir Berhasil Dibuat!


Tentu, berikut adalah laporan eksekutif yang merangkum informasi tersebut dalam sebuah narasi yang kohesif.

***

**Laporan Eksekutif: Profil Peserta**

Analisis terhadap total 2.999 peserta mengungkapkan profil demografis yang spesifik dengan rata-rata usia 36,4 tahun dan didominasi secara signifikan oleh peserta laki-laki (2.427) dibandingkan perempuan (572). Profil ini sebagian besar berasal dari institusi pendidikan teknik terkemuka, dengan ITENAS, Politeknik Negeri Jakarta, dan Politeknik Negeri Semarang menjadi tiga kontributor terbesar. Kecenderungan ini sejalan dengan latar belakang pendidikan mereka, di mana terdapat konsentrasi yang sangat kuat pada bidang teknik, khususnya Teknik Sipil yang secara kolektif (termasuk variasi nama jurusan) menjadi pilihan mayoritas absolut peserta, diikuti oleh Teknik Mesin dan Teknik Lingkungan.

**Penjelasan Kode:**

1. `synthesis_llm.invoke(synthesis_prompt)`: Mengirim prompt laporan ke model AI (di sini menggunakan gemini-2.5-pro yang kemampuannya lebih tinggi untuk tugas penulisan naratif) untuk menghasilkan ringkasan akhir.

2. `display(Markdown(final_report.content))`: Menampilkan hasil laporan dengan format Markdown agar lebih mudah dibaca (misalnya, cetak tebal, miring, dll.).

---

# Langkah 6: Sesi Chatbot Interaktif dengan Data Anda

Cell ini mengubah script menjadi aplikasi interaktif. Anda dapat "mengobrol" dengan data Anda secara langsung dari notebook.

In [19]:
if 'agent_executor' in locals():
    print("👋 Halo! Saya adalah chatbot BigQuery Anda. Apa yang ingin Anda ketahui dari data?")
    print("   Ketik 'exit' atau 'keluar' untuk mengakhiri sesi.")
    print("----------------------------------------------------------------------------------")

    while True:
        try:
            # Meminta input dari pengguna
            natural_language_query = input("❓ Pertanyaan Anda: ")

            # Kondisi untuk keluar dari loop
            if natural_language_query.lower() in ["exit", "keluar", "stop"]:
                print("\n👋 Sampai jumpa lagi!")
                break

            # Cek jika input kosong
            if not natural_language_query:
                print("💬 Mohon masukkan pertanyaan.")
                continue

            print("\n🚀 Agent sedang bekerja, mohon tunggu...")
            # Menjalankan agent dengan query dari pengguna
            result = agent_executor.invoke({"input": natural_language_query})

            print("\n✅ Jawaban Agent:")
            # Mencetak output dari agent dengan penanganan jika tidak ada output
            print(result.get("output", "Maaf, saya tidak dapat menemukan jawaban."))
            print("----------------------------------------------------------------------------------")

        except KeyboardInterrupt:
            print("\n\n👋 Sesi dihentikan oleh pengguna. Sampai jumpa lagi!")
            break
        except Exception as e:
            print(f"\n❌ Terjadi error: {e}")
            print("----------------------------------------------------------------------------------")
else:
    print("❌ Agent belum siap. Harap jalankan sel pada 'Langkah 4' untuk inisialisasi.")

👋 Halo! Saya adalah chatbot BigQuery Anda. Apa yang ingin Anda ketahui dari data?
   Ketik 'exit' atau 'keluar' untuk mengakhiri sesi.
----------------------------------------------------------------------------------
❓ Pertanyaan Anda: Berapa jumlah peserta pelatihan >/

🚀 Agent sedang bekerja, mohon tunggu...


> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{'tool_input': ''}`
responded:  Finally, I will construct the query to answer the question.


tabel_data_pelatihan
Invoking: `sql_db_schema` with `{'table_names': 'tabel_data_pelatihan'}`



CREATE TABLE `tabel_data_pelatihan` (
	`nama_peserta` STRING, 
	`no_ktp` STRING, 
	`jenis_kelamin_peserta` STRING, 
	`usia_peserta` STRING, 
	`pendidikan_peserta` STRING, 
	`jurusan_peserta` STRING, 
	`institusi_peserta` STRING, 
	`email` STRING, 
	`alamat_peserta` STRING, 
	`kabkot_peserta` STRING, 
	`no_hp` STRING, 
	`utusan_peserta` STRING, 
	`nama_jabatan_kerja_peserta` STRING, 
	`jenis_pelatihan` STRING, 

**Penjelasan Kode:**

1. `while True:`: Membuat sebuah infinite loop (perulangan tak terbatas) yang akan terus berjalan sampai dihentikan secara eksplisit.

2. `natural_language_query = input(...)`: Menjeda eksekusi dan menampilkan teks "❓ Pertanyaan Anda: " untuk meminta input dari pengguna. Apa pun yang Anda ketik akan disimpan dalam variabel natural_language_query.

3. `if natural_language_query.lower() in [...]`: Mengecek apakah input pengguna (yang sudah diubah ke huruf kecil) adalah salah satu dari kata kunci untuk keluar (exit, keluar, stop).

4. `break`: Jika kondisi di atas terpenuhi, perintah ini akan menghentikan perulangan while True: dan mengakhiri program.

5. `agent_executor.invoke(...)`: Sama seperti di Langkah 4, ini adalah perintah untuk menjalankan agen dengan pertanyaan yang baru saja Anda masukkan.

6. `try...except...`: Blok ini berfungsi untuk menangani error. Jika terjadi masalah saat agen bekerja (misalnya, tidak bisa terhubung ke internet), program tidak akan berhenti total, melainkan akan menangkap error tersebut, mencetak pesan yang informatif, dan melanjutkan perulangan untuk pertanyaan berikutnya.